# 二吸収帯 Iterative Matched Filter による実HISUIメタンプルーム検出

実HISUIシーンそのものに対してIterative Matched Filterを適用

処理の中心は次の反復

1. 現在の背景候補画素から、各吸収帯の背景平均と共分散を推定する
2. 1.6 µm帯・2.3 µm帯でMatched Filterを計算する
3. 高い正のMF応答を示す画素とその周囲を背景候補から除外する
4. 背景統計を再計算し、除外画素がほぼ変化しなくなるまで繰り返す

各帯域のMF出力は背景分布でrobust標準化

$$
Z_b(x,y)=
\frac{\alpha_b(x,y)-\operatorname{median}_{\mathrm{bg}}(\alpha_b)}
{1.4826\,\operatorname{MAD}_{\mathrm{bg}}(\alpha_b)}
$$

2.3 µm帯を主候補生成に使い、1.6 µm帯を空間的な確認に使う。

- Iterative MFの収束履歴
- 1.6 µm帯・2.3 µm帯のMF応答とrobust Z-score
- 2.3 µm候補のうち、1.6 µm帯が近傍で支持する画素・領域
- 二帯域の重なりが偶然以上かを調べるランダムシフト検定

このNotebookのMF出力 $\alpha$ は、まず**メタン標的方向の応答強度**として解釈する。正確なppm推定値とは限らない。


In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import (
    binary_dilation,
    label,
    maximum_filter,
)

np.set_printoptions(precision=5, suppress=True)


## 1. 設定

In [ ]:
# 入力
ROI_CSV = r"E:\refit\all_roi_spectra.csv"
CH4_LUT_CSV = r"E:\refit\CH4b.csv"
OUTPUT_DIR = Path("./dual_window_iterative_mf_real_plume_output")

# MODTRAN・HISUI設定
BACKGROUND_CH4_PPM = 1.8
FWHM_NM = 12.5
WINDOW_16 = (1580.0, 1750.0)
WINDOW_23 = (2100.0, 2450.0)
UAS_MAX_ENHANCEMENT_PPM = 0.5
UAS_STEP_PPM = 0.05

# Iterative MF設定
MAX_ITERATIONS = 12
EXCLUSION_Z_16 = 2.5
EXCLUSION_Z_23 = 2.5
EXCLUSION_DILATION_PIXELS = 2
MIN_BACKGROUND_FRACTION = 0.55
MIN_BACKGROUND_PIXELS = 300
CONVERGENCE_NEW_PIXEL_FRACTION = 2.5e-4
COVARIANCE_SHRINKAGE = 0.08
COVARIANCE_RIDGE_RELATIVE = 1e-8

# 最終候補抽出
DETECTION_Z_16 = 2.0       # 1.6 µm帯は確認用なので少し緩める
DETECTION_Z_23 = 3.0       # 2.3 µm帯を主候補生成に使う
NEIGHBORHOOD_RADIUS = 1    # 1なら3×3近傍を許容
MIN_REGION_PIXELS = 3
TOP_PIXEL_COUNT = 200

# 二帯域の偶然重なりを調べるランダムシフト検定
SHIFT_TEST_TRIALS = 500
SHIFT_TEST_MIN_PIXELS = 5
RANDOM_SEED = 42

# 任意：図に発生源候補を表示する場合、ROI内インデックス(row, col)を指定
SOURCE_YX_INDEX = None  # 例: (100, 85)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 2. HISUI ROIスペクトルCSVの読み込み

In [ ]:
def get_wave_columns(df):
    pattern = re.compile(r"^wave_([0-9.]+)nm$")
    pairs = []
    for col in df.columns:
        match = pattern.match(str(col))
        if match is not None:
            pairs.append((col, float(match.group(1))))
    if not pairs:
        raise ValueError(
            f"wave_***nm形式の列が見つかりません。先頭列: {list(df.columns[:10])}"
        )
    pairs.sort(key=lambda item: item[1])
    return [p[0] for p in pairs], np.array([p[1] for p in pairs], dtype=float)


def load_roi_spectra_csv(path):
    df = pd.read_csv(path)
    if "y" not in df.columns or "x" not in df.columns:
        raise ValueError("CSVには y と x 列が必要です。")
    wave_cols, wavelengths = get_wave_columns(df)
    spectra = df[wave_cols].to_numpy(dtype=float)
    return df, wavelengths, spectra


def spectra_to_cube(df, spectra, fill_value=np.nan):
    frame = df.reset_index(drop=True)
    ys = np.sort(frame["y"].unique())
    xs = np.sort(frame["x"].unique())
    y_to_i = {value: i for i, value in enumerate(ys)}
    x_to_i = {value: i for i, value in enumerate(xs)}

    cube = np.full(
        (len(ys), len(xs), spectra.shape[1]),
        fill_value,
        dtype=float,
    )
    for row_i, row in frame.iterrows():
        cube[y_to_i[row["y"]], x_to_i[row["x"]]] = spectra[row_i]
    return cube, ys, xs


def make_valid_pixel_mask(
    cube,
    nodata_values=(0.0, -9999.0),
    require_positive=True,
):
    valid = np.isfinite(cube)
    for value in nodata_values:
        valid &= cube != value
    if require_positive:
        valid &= cube > 0
    return np.all(valid, axis=2)


df, wavelengths, spectra = load_roi_spectra_csv(ROI_CSV)
cube_observed, y_values, x_values = spectra_to_cube(df, spectra)
valid_mask = make_valid_pixel_mask(cube_observed)

print("DataFrame shape:", df.shape)
print("Cube shape:", cube_observed.shape)
print("Wavelength range:", wavelengths[0], "to", wavelengths[-1], "nm")
print("Valid pixels:", int(valid_mask.sum()), "/", valid_mask.size)


## 3. MODTRAN絶対濃度LUTの読み込みとUAS作成

In [ ]:
def load_ch4_absolute_concentration_lut(path):
    df_lut = pd.read_csv(path)
    candidates = [
        col for col in df_lut.columns
        if str(col).strip().lower() in {"wavelength", "wave", "wavelength_nm"}
    ]
    if not candidates:
        raise ValueError("LUTに wavelength 列が見つかりません。")

    wave_col = candidates[0]
    mod_wave = df_lut[wave_col].to_numpy(dtype=float)

    pairs = []
    for col in df_lut.columns:
        if col == wave_col:
            continue
        try:
            pairs.append((col, float(str(col).strip())))
        except ValueError:
            pass

    if len(pairs) < 2:
        raise ValueError("LUTに絶対メタン濃度を表す数値列が2列以上必要です。")

    pairs.sort(key=lambda item: item[1])
    concentration_grid = np.array([p[1] for p in pairs], dtype=float)
    lut_spectra = df_lut[[p[0] for p in pairs]].to_numpy(dtype=float).T
    order = np.argsort(mod_wave)
    return mod_wave[order], concentration_grid, lut_spectra[:, order]


def gaussian_srf_resample(
    mod_wave,
    mod_spectra,
    sensor_wave,
    fwhm_nm,
):
    mod_wave = np.asarray(mod_wave, dtype=float)
    mod_spectra = np.asarray(mod_spectra, dtype=float)
    sensor_wave = np.asarray(sensor_wave, dtype=float)

    if np.isscalar(fwhm_nm):
        fwhm = np.full(sensor_wave.shape, float(fwhm_nm))
    else:
        fwhm = np.asarray(fwhm_nm, dtype=float)
    if fwhm.shape != sensor_wave.shape:
        raise ValueError("FWHM配列の長さがsensor_waveと一致しません。")

    output = np.full((mod_spectra.shape[0], sensor_wave.size), np.nan)
    for j, center in enumerate(sensor_wave):
        sigma = fwhm[j] / (2.0 * np.sqrt(2.0 * np.log(2.0)))
        use = np.abs(mod_wave - center) <= 4.0 * sigma

        if use.sum() < 2:
            output[:, j] = np.array([
                np.interp(center, mod_wave, spectrum)
                for spectrum in mod_spectra
            ])
        else:
            weights = np.exp(-0.5 * ((mod_wave[use] - center) / sigma) ** 2)
            weights /= weights.sum()
            output[:, j] = mod_spectra[:, use] @ weights
    return output


def interpolate_lut_spectrum(concentration, concentration_grid, sensor_lut):
    if not concentration_grid.min() <= concentration <= concentration_grid.max():
        raise ValueError(
            f"{concentration:.4f} ppmはLUT範囲外です。"
            f"LUT範囲: {concentration_grid.min():.4f}–{concentration_grid.max():.4f} ppm"
        )
    return np.array([
        np.interp(concentration, concentration_grid, sensor_lut[:, band_i])
        for band_i in range(sensor_lut.shape[1])
    ])


def compute_uas_from_absolute_lut(
    sensor_lut,
    concentration_grid,
    background_ppm,
    max_enhancement_ppm,
    step_ppm,
):
    max_allowed = concentration_grid.max() - background_ppm
    if max_enhancement_ppm > max_allowed:
        raise ValueError(
            f"UAS用増分がLUT範囲を超えます。使用可能最大値: {max_allowed:.4f} ppm"
        )

    enhancement_grid = np.arange(
        0.0,
        max_enhancement_ppm + 0.5 * step_ppm,
        step_ppm,
    )
    enhancement_grid = np.unique(
        np.append(enhancement_grid, max_enhancement_ppm)
    )

    background = interpolate_lut_spectrum(
        background_ppm,
        concentration_grid,
        sensor_lut,
    )
    background = np.maximum(background, 1e-30)

    ratio_rows = []
    for enhancement in enhancement_grid:
        enhanced = interpolate_lut_spectrum(
            background_ppm + enhancement,
            concentration_grid,
            sensor_lut,
        )
        ratio_rows.append(enhanced / background)
    ratio_lut = np.asarray(ratio_rows)

    # ln(T) = intercept - UAS * enhancement として回帰
    design = np.column_stack([
        np.ones_like(enhancement_grid),
        enhancement_grid,
    ])
    log_ratio = np.log(np.maximum(ratio_lut, 1e-30))
    coefficients, _, _, _ = np.linalg.lstsq(
        design,
        log_ratio,
        rcond=None,
    )
    uas = -coefficients[1]
    return uas, enhancement_grid, ratio_lut


modtran_wavelengths, concentration_grid, modtran_spectra = (
    load_ch4_absolute_concentration_lut(CH4_LUT_CSV)
)
sensor_lut_absolute = gaussian_srf_resample(
    modtran_wavelengths,
    modtran_spectra,
    wavelengths,
    FWHM_NM,
)
uas_all, uas_enhancement_grid, uas_ratio_lut = compute_uas_from_absolute_lut(
    sensor_lut_absolute,
    concentration_grid,
    BACKGROUND_CH4_PPM,
    UAS_MAX_ENHANCEMENT_PPM,
    UAS_STEP_PPM,
)

mask_16 = (wavelengths >= WINDOW_16[0]) & (wavelengths <= WINDOW_16[1])
mask_23 = (wavelengths >= WINDOW_23[0]) & (wavelengths <= WINDOW_23[1])

if mask_16.sum() < 2 or mask_23.sum() < 2:
    raise ValueError(
        f"吸収窓内バンド数不足: 1.6 µm={mask_16.sum()}, 2.3 µm={mask_23.sum()}"
    )

print("Absolute CH4 LUT grid [ppm]:", concentration_grid)
print("Bands in 1.6 µm window:", int(mask_16.sum()))
print("Bands in 2.3 µm window:", int(mask_23.sum()))

plt.figure(figsize=(9, 4))
plt.plot(wavelengths[mask_16], uas_all[mask_16], label="1.6 µm UAS")
plt.plot(wavelengths[mask_23], uas_all[mask_23], label="2.3 µm UAS")
plt.xlabel("Wavelength [nm]")
plt.ylabel("UAS [ppm$^{-1}$]")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


## 4. Iterative MFの関数

背景候補画素から平均 $\boldsymbol{\mu}$ と正則化共分散 $\boldsymbol{\Sigma}$ を求め、標的を

$$
\boldsymbol{t}=-\boldsymbol{\mu}\odot\boldsymbol{s}
$$

としてMFを計算

$$
\alpha_i=
\frac{\boldsymbol{t}^{\mathsf T}\boldsymbol{\Sigma}^{-1}
(\boldsymbol{x}_i-\boldsymbol{\mu})}
{\boldsymbol{t}^{\mathsf T}\boldsymbol{\Sigma}^{-1}\boldsymbol{t}}
$$

1.6 µm帯と2.3 µm帯は同じ背景候補マスクを共有。どちらかの帯域で高い正の異常を示す画素と、その周囲を次回の背景統計から除外。


In [ ]:
def robust_location_scale(values, floor=1e-12):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size < 2:
        raise ValueError("robust統計量の計算に必要な値が不足しています。")

    center = float(np.median(values))
    mad = float(np.median(np.abs(values - center)))
    scale = 1.4826 * mad

    if not np.isfinite(scale) or scale <= floor:
        scale = float(np.std(values, ddof=1))
    if not np.isfinite(scale) or scale <= floor:
        raise ValueError("MF背景分布の分散がほぼ0です。")
    return center, scale


def regularized_covariance(
    samples,
    shrinkage=0.08,
    ridge_relative=1e-8,
):
    samples = np.asarray(samples, dtype=float)
    if samples.ndim != 2:
        raise ValueError("samplesは[画素, バンド]の2次元配列にしてください。")
    if samples.shape[0] < 2:
        raise ValueError("共分散推定に必要な背景画素が不足しています。")

    covariance = np.cov(samples, rowvar=False, ddof=1)
    covariance = np.atleast_2d(covariance)
    n_bands = covariance.shape[0]

    average_variance = float(np.trace(covariance) / max(n_bands, 1))
    if not np.isfinite(average_variance) or average_variance <= 0:
        diagonal = np.nanvar(samples, axis=0, ddof=1)
        good = diagonal[np.isfinite(diagonal) & (diagonal > 0)]
        average_variance = float(np.median(good)) if good.size else 1.0

    covariance = (
        (1.0 - shrinkage) * covariance
        + shrinkage * average_variance * np.eye(n_bands)
    )
    covariance += (
        ridge_relative * max(average_variance, 1e-30) * np.eye(n_bands)
    )
    return covariance


def fit_global_mf(
    cube,
    band_mask,
    uas,
    valid_mask,
    background_mask,
    covariance_shrinkage,
    covariance_ridge_relative,
):
    band_cube = np.asarray(cube[:, :, band_mask], dtype=float)
    height, width, n_bands = band_cube.shape
    flat = band_cube.reshape(-1, n_bands)

    valid_flat = valid_mask.ravel()
    background_flat = background_mask.ravel()
    background_samples = flat[background_flat]

    minimum_samples = max(MIN_BACKGROUND_PIXELS, 5 * n_bands)
    if background_samples.shape[0] < minimum_samples:
        raise ValueError(
            f"背景画素が不足しています: {background_samples.shape[0]} < {minimum_samples}"
        )

    background_mean = np.mean(background_samples, axis=0)
    covariance = regularized_covariance(
        background_samples,
        shrinkage=covariance_shrinkage,
        ridge_relative=covariance_ridge_relative,
    )

    target = -background_mean * np.asarray(uas[band_mask], dtype=float)
    try:
        inverse_cov_target = np.linalg.solve(covariance, target)
    except np.linalg.LinAlgError:
        warnings.warn("共分散行列のsolveに失敗したため疑似逆行列を使用します。")
        inverse_cov_target = np.linalg.pinv(covariance) @ target

    denominator = float(target @ inverse_cov_target)
    if not np.isfinite(denominator) or denominator <= 1e-30:
        raise ValueError("MF分母が0または不正です。UAS・波長窓・共分散を確認してください。")

    alpha_flat = np.full(flat.shape[0], np.nan, dtype=float)
    residual_valid = flat[valid_flat] - background_mean
    alpha_flat[valid_flat] = residual_valid @ inverse_cov_target / denominator
    alpha = alpha_flat.reshape(height, width)

    center, scale = robust_location_scale(alpha[background_mask])
    zscore = (alpha - center) / scale
    zscore[~valid_mask] = np.nan

    alpha_standard_error = 1.0 / np.sqrt(denominator)
    return {
        "alpha": alpha,
        "zscore": zscore,
        "background_mean": background_mean,
        "covariance": covariance,
        "target": target,
        "alpha_center": center,
        "alpha_scale": scale,
        "alpha_standard_error": alpha_standard_error,
        "denominator": denominator,
    }


def iterative_dual_window_mf(
    cube,
    uas,
    mask_16,
    mask_23,
    valid_mask,
    max_iterations=12,
    exclusion_z_16=2.5,
    exclusion_z_23=2.5,
    dilation_pixels=2,
    minimum_background_fraction=0.55,
    convergence_new_pixel_fraction=2.5e-4,
    covariance_shrinkage=0.08,
    covariance_ridge_relative=1e-8,
):
    valid_count = int(valid_mask.sum())
    if valid_count < MIN_BACKGROUND_PIXELS:
        raise ValueError("有効画素数が少なすぎます。")

    excluded_mask = np.zeros_like(valid_mask, dtype=bool)
    background_mask = valid_mask.copy()
    history_rows = []

    final_16 = None
    final_23 = None

    for iteration in range(max_iterations):
        result_16 = fit_global_mf(
            cube,
            mask_16,
            uas,
            valid_mask,
            background_mask,
            covariance_shrinkage,
            covariance_ridge_relative,
        )
        result_23 = fit_global_mf(
            cube,
            mask_23,
            uas,
            valid_mask,
            background_mask,
            covariance_shrinkage,
            covariance_ridge_relative,
        )

        high_response = valid_mask & (
            (result_16["zscore"] >= exclusion_z_16)
            | (result_23["zscore"] >= exclusion_z_23)
        )

        if dilation_pixels > 0:
            high_response = binary_dilation(
                high_response,
                iterations=dilation_pixels,
            ) & valid_mask

        proposed_excluded = excluded_mask | high_response
        proposed_background = valid_mask & (~proposed_excluded)

        minimum_background_count = max(
            MIN_BACKGROUND_PIXELS,
            int(np.ceil(minimum_background_fraction * valid_count)),
        )
        if int(proposed_background.sum()) < minimum_background_count:
            warnings.warn(
                "除外すると背景画素が設定下限を下回るため、直前の背景マスクで停止します。"
            )
            final_16 = result_16
            final_23 = result_23
            break

        newly_excluded = proposed_excluded & (~excluded_mask)
        new_fraction = float(newly_excluded.sum() / valid_count)

        history_rows.append({
            "iteration": iteration,
            "background_pixels": int(background_mask.sum()),
            "excluded_pixels_before_update": int(excluded_mask.sum()),
            "candidate_pixels_before_dilation": int((valid_mask & (
                (result_16["zscore"] >= exclusion_z_16)
                | (result_23["zscore"] >= exclusion_z_23)
            )).sum()),
            "excluded_pixels_after_update": int(proposed_excluded.sum()),
            "newly_excluded_pixels": int(newly_excluded.sum()),
            "newly_excluded_fraction": new_fraction,
            "alpha_center_16": result_16["alpha_center"],
            "alpha_scale_16": result_16["alpha_scale"],
            "alpha_center_23": result_23["alpha_center"],
            "alpha_scale_23": result_23["alpha_scale"],
        })

        excluded_mask = proposed_excluded
        background_mask = proposed_background
        final_16 = result_16
        final_23 = result_23

        print(
            f"iteration={iteration:02d}, "
            f"background={int(background_mask.sum())}, "
            f"excluded={int(excluded_mask.sum())}, "
            f"new={int(newly_excluded.sum())}"
        )

        if new_fraction <= convergence_new_pixel_fraction:
            print("Converged: newly excluded pixel fraction is sufficiently small.")
            break

    # 最終背景マスクでMFを再計算
    final_16 = fit_global_mf(
        cube,
        mask_16,
        uas,
        valid_mask,
        background_mask,
        covariance_shrinkage,
        covariance_ridge_relative,
    )
    final_23 = fit_global_mf(
        cube,
        mask_23,
        uas,
        valid_mask,
        background_mask,
        covariance_shrinkage,
        covariance_ridge_relative,
    )

    return {
        "result_16": final_16,
        "result_23": final_23,
        "background_mask": background_mask,
        "excluded_mask": excluded_mask,
        "history": pd.DataFrame(history_rows),
    }


## 5. 実シーンへIterative MFを適用

In [ ]:
iterative_result = iterative_dual_window_mf(
    cube_observed,
    uas_all,
    mask_16,
    mask_23,
    valid_mask,
    max_iterations=MAX_ITERATIONS,
    exclusion_z_16=EXCLUSION_Z_16,
    exclusion_z_23=EXCLUSION_Z_23,
    dilation_pixels=EXCLUSION_DILATION_PIXELS,
    minimum_background_fraction=MIN_BACKGROUND_FRACTION,
    convergence_new_pixel_fraction=CONVERGENCE_NEW_PIXEL_FRACTION,
    covariance_shrinkage=COVARIANCE_SHRINKAGE,
    covariance_ridge_relative=COVARIANCE_RIDGE_RELATIVE,
)

mf_result_16 = iterative_result["result_16"]
mf_result_23 = iterative_result["result_23"]
iterative_background_mask = iterative_result["background_mask"]
iterative_excluded_mask = iterative_result["excluded_mask"]
convergence_df = iterative_result["history"]

alpha_mf_16 = mf_result_16["alpha"]
alpha_mf_23 = mf_result_23["alpha"]
z_mf_16 = mf_result_16["zscore"]
z_mf_23 = mf_result_23["zscore"]

print("Final background pixels:", int(iterative_background_mask.sum()))
print("Final excluded pixels:", int(iterative_excluded_mask.sum()))
print("1.6 µm alpha standard error approximation:", mf_result_16["alpha_standard_error"])
print("2.3 µm alpha standard error approximation:", mf_result_23["alpha_standard_error"])

display(convergence_df)


## 6. 二吸収帯の候補抽出と空間整合

2.3 µm帯を主候補生成に使い、1.6 µm帯を確認に使う

- `candidate_23`：2.3 µm帯で高応答
- `dual_exact_mask`：同じ画素で両帯域がしきい値超過
- `dual_nearby_mask`：2.3 µm候補の近傍で1.6 µm帯がしきい値超過
- `dual_min_score`：両帯域のうち低いZ-score


In [ ]:
def remove_small_components(binary_mask, minimum_pixels=1):
    component_labels, number_of_components = label(binary_mask)
    if number_of_components == 0:
        return np.zeros_like(binary_mask, dtype=bool), component_labels

    counts = np.bincount(component_labels.ravel())
    keep_ids = np.flatnonzero(counts >= minimum_pixels)
    keep_ids = keep_ids[keep_ids != 0]
    cleaned = np.isin(component_labels, keep_ids)
    return cleaned, component_labels


def neighborhood_supported_mask(
    primary_mask,
    support_score,
    support_threshold,
    radius=1,
):
    if radius < 0:
        raise ValueError("radiusは0以上にしてください。")

    finite_support = np.where(
        np.isfinite(support_score),
        support_score,
        -np.inf,
    )
    local_support_max = maximum_filter(
        finite_support,
        size=2 * radius + 1,
        mode="nearest",
    )
    confirmed = primary_mask & (local_support_max >= support_threshold)
    return confirmed, local_support_max


candidate_16_raw = valid_mask & (z_mf_16 >= DETECTION_Z_16)
candidate_23_raw = valid_mask & (z_mf_23 >= DETECTION_Z_23)

candidate_16, _ = remove_small_components(
    candidate_16_raw,
    MIN_REGION_PIXELS,
)
candidate_23, _ = remove_small_components(
    candidate_23_raw,
    MIN_REGION_PIXELS,
)

dual_exact_mask = candidate_16 & candidate_23
dual_nearby_mask, local_max_z16 = neighborhood_supported_mask(
    candidate_23,
    z_mf_16,
    DETECTION_Z_16,
    radius=NEIGHBORHOOD_RADIUS,
)

dual_min_score = np.minimum(z_mf_16, z_mf_23)
dual_geometric_score = np.sqrt(
    np.clip(z_mf_16, 0.0, None)
    * np.clip(z_mf_23, 0.0, None)
)
dual_min_score[~valid_mask] = np.nan
dual_geometric_score[~valid_mask] = np.nan

print("1.6 µm candidates:", int(candidate_16.sum()))
print("2.3 µm candidates:", int(candidate_23.sum()))
print("Exact dual-window candidates:", int(dual_exact_mask.sum()))
print("Nearby-confirmed candidates:", int(dual_nearby_mask.sum()))


## 7. 2.3 µm候補領域を1.6 µm帯で確認

In [ ]:
def summarize_primary_regions(
    primary_mask,
    z_primary,
    z_support,
    alpha_primary,
    alpha_support,
    y_values,
    x_values,
    support_threshold,
    neighborhood_support,
):
    region_labels, number_of_regions = label(primary_mask)
    rows = []

    for region_id in range(1, number_of_regions + 1):
        region = region_labels == region_id
        yy, xx = np.where(region)
        if yy.size == 0:
            continue

        primary_values = z_primary[region]
        support_values = z_support[region]
        nearby_values = neighborhood_support[region]

        peak_local_index = int(np.nanargmax(primary_values))
        peak_row = int(yy[peak_local_index])
        peak_col = int(xx[peak_local_index])

        rows.append({
            "region_id": region_id,
            "pixel_count": int(region.sum()),
            "centroid_row_index": float(np.mean(yy)),
            "centroid_col_index": float(np.mean(xx)),
            "peak_row_index": peak_row,
            "peak_col_index": peak_col,
            "peak_y_coordinate": y_values[peak_row],
            "peak_x_coordinate": x_values[peak_col],
            "max_z23": float(np.nanmax(primary_values)),
            "median_z23": float(np.nanmedian(primary_values)),
            "max_z16_same_region": float(np.nanmax(support_values)),
            "median_z16_same_region": float(np.nanmedian(support_values)),
            "max_z16_nearby": float(np.nanmax(nearby_values)),
            "max_alpha23": float(np.nanmax(alpha_primary[region])),
            "max_alpha16_same_region": float(np.nanmax(alpha_support[region])),
            "supported_same_region": bool(
                np.nanmax(support_values) >= support_threshold
            ),
            "supported_with_neighborhood": bool(
                np.nanmax(nearby_values) >= support_threshold
            ),
        })

    result = pd.DataFrame(rows)
    if not result.empty:
        result = result.sort_values(
            ["supported_with_neighborhood", "max_z23", "pixel_count"],
            ascending=[False, False, False],
        ).reset_index(drop=True)
    return result, region_labels


region_df, candidate_23_region_labels = summarize_primary_regions(
    candidate_23,
    z_mf_23,
    z_mf_16,
    alpha_mf_23,
    alpha_mf_16,
    y_values,
    x_values,
    DETECTION_Z_16,
    local_max_z16,
)

display(region_df)


## 8. 二帯域の重なりに対するランダムシフト検定

実プルームには正解マスクがないため、二帯域の候補が偶然以上に重なっているかを補助的に調べる。1.6 µm候補マスクをランダムに平行移動し、2.3 µm候補との近傍重なり画素数の帰無分布を作る。


In [ ]:
def nearby_overlap_count(primary_mask, support_mask, radius):
    support_nearby = maximum_filter(
        support_mask.astype(np.uint8),
        size=2 * radius + 1,
        mode="wrap",
    ) > 0
    return int(np.sum(primary_mask & support_nearby))


def random_shift_overlap_test(
    primary_mask,
    support_mask,
    valid_mask,
    radius=1,
    trials=500,
    minimum_shift_pixels=5,
    random_seed=42,
):
    rng = np.random.default_rng(random_seed)
    height, width = primary_mask.shape

    observed = nearby_overlap_count(primary_mask, support_mask, radius)
    null_counts = []

    maximum_dy = max(1, height // 2)
    maximum_dx = max(1, width // 2)

    attempts = 0
    while len(null_counts) < trials and attempts < 20 * trials:
        attempts += 1
        dy = int(rng.integers(-maximum_dy, maximum_dy + 1))
        dx = int(rng.integers(-maximum_dx, maximum_dx + 1))
        if abs(dy) < minimum_shift_pixels and abs(dx) < minimum_shift_pixels:
            continue

        shifted_support = np.roll(support_mask, shift=(dy, dx), axis=(0, 1))
        shifted_valid = np.roll(valid_mask, shift=(dy, dx), axis=(0, 1))
        comparison_valid = valid_mask & shifted_valid

        count = nearby_overlap_count(
            primary_mask & comparison_valid,
            shifted_support & comparison_valid,
            radius,
        )
        null_counts.append(count)

    null_counts = np.asarray(null_counts, dtype=int)
    if null_counts.size == 0:
        return {
            "observed_overlap": observed,
            "null_mean": np.nan,
            "null_std": np.nan,
            "p_value": np.nan,
            "null_counts": null_counts,
        }

    p_value = (1 + np.sum(null_counts >= observed)) / (null_counts.size + 1)
    return {
        "observed_overlap": observed,
        "null_mean": float(np.mean(null_counts)),
        "null_std": float(np.std(null_counts, ddof=1)) if null_counts.size > 1 else 0.0,
        "p_value": float(p_value),
        "null_counts": null_counts,
    }


shift_test = random_shift_overlap_test(
    candidate_23,
    candidate_16,
    valid_mask,
    radius=NEIGHBORHOOD_RADIUS,
    trials=SHIFT_TEST_TRIALS,
    minimum_shift_pixels=SHIFT_TEST_MIN_PIXELS,
    random_seed=RANDOM_SEED,
)

shift_test_summary_df = pd.DataFrame([{
    "observed_nearby_overlap_pixels": shift_test["observed_overlap"],
    "null_mean_overlap_pixels": shift_test["null_mean"],
    "null_std_overlap_pixels": shift_test["null_std"],
    "random_shift_p_value": shift_test["p_value"],
    "trials": int(shift_test["null_counts"].size),
}])

display(shift_test_summary_df)

if shift_test["null_counts"].size:
    plt.figure(figsize=(6, 4))
    plt.hist(shift_test["null_counts"], bins=30)
    plt.axvline(
        shift_test["observed_overlap"],
        linestyle="--",
        linewidth=2,
        label="Observed overlap",
    )
    plt.xlabel("Nearby overlap pixels after random shift")
    plt.ylabel("Count")
    plt.title("Random-shift overlap test")
    plt.legend()
    plt.tight_layout()
    plt.show()
